In [1]:
import numpy as np

# Here we have homework-1 instead of homework-2 due to an off-by-one error
train = np.load("/kaggle/input/competitions/nn-2026-fii-homework-1/train.npz")
test = np.load("/kaggle/input/competitions/nn-2026-fii-homework-1/test.npz")

X_train = train["data"]
y_train = train["labels"]
X_test = test["data"]

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)


(60000, 784)
(60000,)
(15000, 784)


In [2]:
import pandas as pd

# Scale pixels from [0, 255] to [0, 1].
X_train = train["data"].astype(np.float32) / 255.0
X_test = test["data"].astype(np.float32) / 255.0
print("scaled pixel range:", X_train.min(), X_train.max())

NUM_CLASSES = 10

def one_hot(labels, num_classes=NUM_CLASSES):
    T = np.zeros((labels.shape[0], num_classes), dtype=np.float32)
    T[np.arange(labels.shape[0]), labels] = 1.0
    return T

T_train = one_hot(y_train)
print(X_train.shape, T_train.shape, X_test.shape)


scaled pixel range: 0.0 1.0
(60000, 784) (60000, 10) (15000, 784)


In [3]:
# forward propagation: z = W·x + b
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True) # avoids overflow, same result
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

def forward(X, W, b):
    Z = X @ W + b # (m, 784) @ (784, 10) + (10,) -> (m, 10)
    return softmax(Z) # class probabilities

def cross_entropy(Y, T):
    return -np.mean(np.sum(T * np.log(Y + 1e-12), axis=1))

def backward(X, T, Y, W, b, lr):
    m = X.shape[0]
    error = T - Y                      # (Target - y)
    W += lr * (X.T @ error) / m        # W = W + lr * x^T (Target - y)
    b += lr * error.mean(axis=0)       # b = b + lr * (Target - y)
    return W, b

# training
def predict(X, W, b):
    return np.argmax(forward(X, W, b), axis=1)

def accuracy(X, labels, W, b):
    return np.mean(predict(X, W, b) == labels)

def train_model(X, T, labels, X_val=None, labels_val=None,
                epochs=50, batch_size=100, lr=0.5, seed=986, validation=True):
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 0.01, size=(X.shape[1], NUM_CLASSES)).astype(np.float32)
    b = np.zeros(NUM_CLASSES, dtype=np.float32)
    m = X.shape[0]

    for epoch in range(1, epochs + 1):
        lr_epoch = lr * 0.5 ** ((epoch - 1) // 10) # decrease learning rate every 10 epochs
        perm = rng.permutation(m) # shuffle each epoch
        for start in range(0, m, batch_size):
            idx = perm[start:start + batch_size]
            Xb, Tb = X[idx], T[idx]
            Yb = forward(Xb, W, b)
            W, b = backward(Xb, Tb, Yb, W, b, lr_epoch)

        if epoch % 5 == 0:
            loss = cross_entropy(forward(X, W, b), T)
            msg = f"{epoch}. lr {lr_epoch:.4f} | training accuracy {accuracy(X, labels, W, b):.4f}"
            if validation == True:
                msg+=f"| accuracy {accuracy(X_val, labels_val, W, b):.4f}"
            print(msg)
    return W, b

# Validation
rng = np.random.default_rng(0)
perm = rng.permutation(X_train.shape[0])
n_val = X_train.shape[0] // 10
val_idx, tr_idx = perm[:n_val], perm[n_val:]

# W, b = train_model(X_train[tr_idx], T_train[tr_idx], y_train[tr_idx],
#                    X_train[val_idx], y_train[val_idx],
#                    epochs=50, batch_size=100, lr=0.5, validation=True)

# Final training with all data
W, b = train_model(X_train, T_train, y_train, epochs=50, batch_size=100, lr=0.5, validation = False)

predictions = predict(X_test, W, b)
df = pd.DataFrame({"ID": np.arange(len(predictions)), "target": predictions.astype(int)})
df.to_csv("submission.csv", index=False)

5. lr 0.5000 | training accuracy 0.9236
10. lr 0.5000 | training accuracy 0.9296
15. lr 0.2500 | training accuracy 0.9319
20. lr 0.2500 | training accuracy 0.9303
25. lr 0.1250 | training accuracy 0.9332
30. lr 0.1250 | training accuracy 0.9322
35. lr 0.0625 | training accuracy 0.9337
40. lr 0.0625 | training accuracy 0.9341
45. lr 0.0312 | training accuracy 0.9339
50. lr 0.0312 | training accuracy 0.9343
